In [ ]:
#| default_exp shortcuts.multiple_notes

In [ ]:
#| export
from os import PathLike
from typing import Literal

from trouver.obsidian.file import MarkdownFile
from trouver.obsidian.vault import VaultNote
from trouver.personal_vault.note_processing import process_standard_information_note

from trouver.obsidian.links import links_from_text

In [ ]:
from unittest.mock import patch, MagicMock
from fastcore.test import test_eq, test_fail

## Getting notes linked in a `str`

In [ ]:
#| export
def notes_from_names(
        vault: PathLike,
        note_names: list[str],
        ) -> list[VaultNote]:
    return [VaultNote(vault=vault, name=name) for name in note_names]

In [ ]:
#| hide
from fastcore.test import test_eq
from unittest.mock import patch, MagicMock
from os import PathLike

# --- Tests ---
# We patch the 'VaultNote' class where it is defined/imported
with patch('__main__.VaultNote') as MockNote:
    # Setup
    test_vault = "/mock/path"
    test_names = ["Note1", "Note2"]
    
    # Run
    res = notes_from_names(test_vault, test_names)
    
    # Assertions using fastcore
    test_eq(len(res), 2)
    
    # Verify the calls were made with correct arguments
    # MockNote.call_args_list returns a list of call objects
    test_eq(MockNote.call_args_list[0].kwargs['name'], "Note1")
    test_eq(MockNote.call_args_list[0].kwargs['vault'], test_vault)
    test_eq(MockNote.call_args_list[1].kwargs['name'], "Note2")

In [ ]:
#| export
def notes_from_links_in_text(
        vault: PathLike,
        text_with_links: str,
        ) -> list[VaultNote]:
    links = links_from_text(text_with_links)
    note_names: list[str] = [link.file_name for link in links]
    return notes_from_names(vault, note_names)

In [ ]:
#| hide
# --- Tests ---
# We mock the two functions this one relies on
with patch('__main__.links_from_text') as mock_links, \
     patch('__main__.notes_from_names') as mock_notes:
    
    # 1. Setup Mock Data
    # Create a fake 'Link' object that has a .file_name attribute
    mock_link = MagicMock()
    mock_link.file_name = "TargetNote"
    mock_links.return_value = [mock_link]
    
    # Create a fake return for the final step
    mock_notes.return_value = ["FakeVaultNoteObject"]
    
    # 2. Run the function
    vault_path = "/my/vault"
    input_text = "Check out [[TargetNote]]"
    res = notes_from_links_in_text(vault_path, input_text)
    
    # 3. Fastcore Assertions
    # Did it call the parser with our text?
    test_eq(mock_links.call_args[0][0], input_text)
    
    # Did it pass the extracted name to the note creator?
    # We check that the second argument (note_names) was ['TargetNote']
    test_eq(mock_notes.call_args[0][1], ["TargetNote"])
    
    # Did it return what notes_from_names gave it?
    test_eq(res, ["FakeVaultNoteObject"])

## Processing multiple notes at once

In [ ]:
#| export
def process_multiple_notes(
    notes: list[VaultNote] | str, # Either a list of `VaultNote` objects or a str with links to notes. 
    vault: PathLike | None = None,
    as_single_str: bool = True,
    separator: str = '\n\n\n',
    **kwargs, # arguments to pass to `process_standard_information_note`
) -> list[str] | str:
    # Handle string input (extracting notes from links)
    if isinstance(notes, str):
        if vault is None:
            raise ValueError("A 'vault' path must be provided when passing notes as a string.")
        notes = notes_from_links_in_text(vault, notes)

    if not notes:
        return '' if as_single_str else []
    
    # Infer vault from first note if not explicitly provided
    if vault is None:
        vault = notes[0].vault

    processed = [
        str(process_standard_information_note(note.text(), vault, **kwargs)) 
        for note in notes
    ]
    
    return separator.join(processed) if as_single_str else processed

In [ ]:
#| hide

# --- Tests ---
with patch('__main__.process_standard_information_note') as mock_process:
    # 1. Setup Mock Notes
    # We create two mock notes with different text but the same vault
    mock_vault = "/fake/vault"
    
    note1 = MagicMock()
    note1.text.return_value = "Content 1"
    note1.vault = mock_vault
    
    note2 = MagicMock()
    note2.text.return_value = "Content 2"
    note2.vault = mock_vault
    
    notes_list = [note1, note2]
    
    # Define what the processing function returns for each note
    mock_process.side_effect = ["Processed 1", "Processed 2"]
    
    # 2. Test Case A: Return as single string (Default)
    res_str = process_multiple_notes(notes_list, separator=" | ", custom_arg=True)
    
    test_eq(res_str, "Processed 1 | Processed 2")
    # Verify kwargs were passed through
    test_eq(mock_process.call_args.kwargs['custom_arg'], True)
    # Verify vault was inferred correctly from the first note
    test_eq(mock_process.call_args.args[1], mock_vault)

    # 3. Test Case B: Return as list
    # Reset side effect for fresh run
    mock_process.side_effect = ["Processed 1", "Processed 2"]
    res_list = process_multiple_notes(notes_list, as_single_str=False)
    
    test_eq(res_list, ["Processed 1", "Processed 2"])
    test_eq(len(res_list), 2)

    # 4. Test Case C: Empty Input
    test_eq(process_multiple_notes([], as_single_str=True), "")
    test_eq(process_multiple_notes([], as_single_str=False), [])

In [ ]:
#| hide


with patch('__main__.notes_from_links_in_text') as mock_extract, \
     patch('__main__.process_standard_information_note') as mock_process:
    
    # Setup: What happens when we pass a string?
    mock_vault = "/fake/vault"
    input_str = "[[Note1]] and [[Note2]]"
    
    # Create mock notes to be "returned" by the extractor
    n1, n2 = MagicMock(), MagicMock()
    n1.text.return_value = "Content 1"
    n2.text.return_value = "Content 2"
    mock_extract.return_value = [n1, n2]
    
    mock_process.side_effect = ["Result 1", "Result 2"]

    # --- Test 1: Successful String Processing ---
    res = process_multiple_notes(input_str, vault=mock_vault, as_single_str=False)
    
    # Did it call the extractor with the right string and vault?
    test_eq(mock_extract.call_args.args, (mock_vault, input_str))
    test_eq(res, ["Result 1", "Result 2"])

    # --- Test 2: Error when no vault is provided for a string ---
    # Using fastcore's test_fail to ensure it raises ValueError
    test_fail(lambda: process_multiple_notes(input_str, vault=None), contains="must be provided")
    
    # --- Test 3: Empty string (No links found) ---
    mock_extract.return_value = []
    test_eq(process_multiple_notes("No links here", vault=mock_vault), "")

## Only consider the notes after the specified one in a list

In [ ]:
#| export

def trailing_or_leading_subset_notes(
    notes: list[VaultNote],
    target: VaultNote | str, # A VaultNote or name thereof
    direction: Literal['after', 'before'] = 'after', # 'after' or 'before'
    include_target: bool = False
) -> list[VaultNote]:
    """Returns a sublist of notes relative to a target note or note name."""
    
    # Extract name if a VaultNote object was passed
    target_name = target.name if hasattr(target, 'name') else target
    
    # Find the index of the target
    try:
        idx = next(i for i, n in enumerate(notes) if n.name == target_name)
    except StopIteration:
        raise ValueError(f"Note '{target_name}' not found in the provided list.")

    if direction == 'before':
        end_idx = idx + 1 if include_target else idx
        return notes[:end_idx]
    
    elif direction == 'after':
        start_idx = idx if include_target else idx + 1
        return notes[start_idx:]
    
    else:
        raise ValueError("direction must be either 'before' or 'after'")

In [ ]:
#| hide


# Setup mock notes
def _mk_note(name):
    n = MagicMock()
    n.name = name
    return n

n1, n2, n3, n4 = [_mk_note(f"Note{i}") for i in range(1, 5)]
notes_list = [n1, n2, n3, n4]

# --- Test: Basic 'after' functionality ---
# After 'Note2' (exclude) -> [Note3, Note4]
test_eq(trailing_or_leading_subset_notes(notes_list, "Note2", direction='after'), [n3, n4])

# --- Test: 'after' with inclusion ---
# After 'Note2' (include) -> [Note2, Note3, Note4]
test_eq(trailing_or_leading_subset_notes(notes_list, "Note2", direction='after', include_target=True), [n2, n3, n4])

# --- Test: 'before' functionality ---
# Before 'Note3' (exclude) -> [Note1, Note2]
test_eq(trailing_or_leading_subset_notes(notes_list, "Note3", direction='before'), [n1, n2])

# --- Test: 'before' with inclusion ---
# Before 'Note3' (include) -> [Note1, Note2, Note3]
test_eq(trailing_or_leading_subset_notes(notes_list, "Note3", direction='before', include_target=True), [n1, n2, n3])

# --- Test: Passing a VaultNote object instead of a string ---
test_eq(trailing_or_leading_subset_notes(notes_list, n2, direction='after'), [n3, n4])

# --- Test: Error Handling ---
test_fail(lambda: trailing_or_leading_subset_notes(notes_list, "MissingNote"), contains="not found")
test_fail(lambda: trailing_or_leading_subset_notes(notes_list, "Note1", direction="sideways"), contains="direction must be")